# Imports

In [ ]:
import torch
import os
import pickle
from model.autoencoder import Model
from data import get_cross_data, get_rec_data, load_data
import numpy as np
import sys
import traceback

In [ ]:
unpaired_batch_size = 32
paired_batch_size = int(unpaired_batch_size/4)
T = 64
M = 1           # Number of persons
V = 25          # Number of joints
setting = 'cs'  # 'cs' or 'cv'
dataset = 'ntu120'
datasets = {
    'ntu120': {
        'num_classes': 120,
        'num_actors': 106,
        'graph': 'graph.ntu_rgb_d.Graph'
    },
    'ntu': {
        'num_classes': 60,
        'num_actors': 40,
        'graph': 'graph.ntu_rgb_d.Graph'
    },
}

# Data

In [ ]:
# Load data
X = load_data(dataset)
use_cache = True

# Try to load paired data from pickle file
if use_cache and os.path.exists(f'data/{dataset}_{setting}_paired.pkl'):
    with open(f'data/{dataset}_{setting}_paired.pkl', 'rb') as f:
        paired_data = pickle.load(f)
        paired_train = paired_data['train']
        paired_test = paired_data['test']
        print('Paired data loaded from pickle file')
else:
    # Generate paired data and save to pickle file
    paired_train, paired_test = get_cross_data(X, dataset, setting, paired_batch_size, T, return_loader=True, train_samples=64, test_samples=32)
    if use_cache:
        with open(f'data/{dataset}_{setting}_paired.pkl', 'wb') as f:
            pickle.dump({'train': paired_train, 'test': paired_test}, f)
            print('Paired data saved to pickle file')

# Generate unpaired data
train, test = get_rec_data(X, dataset, setting, T, unpaired_batch_size)

# Transformer Retargeting

In [ ]:
# Initialize the model
model = Model(num_class= 120 if dataset == 'ntu120' else 60, num_point=V, num_person=M, graph='graph.ntu_rgb_d.Graph',
              graph_args={'labeling_mode': 'spatial'}, debug=False)
model = model.cuda()

model.load_state_dict(torch.load('model.pth'))

## Anonymization

In [ ]:
def prep_data(x):
    N = x.shape[0]
    T = x.shape[1]
    D = x.shape[2]
    M = 1
    V = 25
    C_in = 3

    # Ensure that M * V * C_in == D
    assert M * V * C_in == D, "Mismatch in dimensions"

    # Reshape inputs
    x = x.view(N, T, M, V, C_in).permute(0, 4, 1, 3, 2).contiguous()
    
    return x

def get_anonymized_paired(batch, model, prep_data):
    anonymized = [] # {'skeleton': skeleton, 'retargeted_actor': int, 'original_actor': int, 'action': int}

    x1, x2, y1, y2, actors, actions = batch
    N = x1.shape[0]
    T = x1.shape[1]
    D = x1.shape[2]
    
    # Prepare data
    x1 = prep_data(x1).cuda()
    x2 = prep_data(x2).cuda()
    y1 = prep_data(y1).cuda()
    y2 = prep_data(y2).cuda()

    # Concatenate inputs along the batch dimension
    all_inputs = torch.cat([x1, x2, y1, y2], dim=0)
    all_targets = torch.cat([x2, x1, y2, y1], dim=0)

    # Call the model once with the concatenated inputs
    all_outputs = model(all_inputs, all_targets)

    # Split the outputs back into the original components
    x1_hat, x2_hat, y1_hat, y2_hat = torch.split(all_outputs, N, dim=0)

    # Reshape and permute
    x1_hat = x1_hat.permute(0, 2, 4, 3, 1).contiguous().view(N, T, D)
    x2_hat = x2_hat.permute(0, 2, 4, 3, 1).contiguous().view(N, T, D)
    y1_hat = y1_hat.permute(0, 2, 4, 3, 1).contiguous().view(N, T, D)
    y2_hat = y2_hat.permute(0, 2, 4, 3, 1).contiguous().view(N, T, D)

    
    for i in range(N):
        # x1_hat corresponds to retargeted_actor = actors[i, 1], original_actor = actors[i, 0]
        anonymized.append({
            'skeleton': x1_hat[i],
            'retargeted_actor': actors[i, 1],
            'original_actor': actors[i, 0],
            'action': actions[i, 0]
        })

        # x2_hat corresponds to retargeted_actor = actors[i, 0], original_actor = actors[i, 1]
        anonymized.append({
            'skeleton': x2_hat[i],
            'retargeted_actor': actors[i, 0],
            'original_actor': actors[i, 1],
            'action': actions[i, 1]
        })

        # y1_hat corresponds to retargeted_actor = actors[i, 1], original_actor = actors[i, 0]
        anonymized.append({
            'skeleton': y1_hat[i],
            'retargeted_actor': actors[i, 1],
            'original_actor': actors[i, 0],
            'action': actions[i, 0]
        })

        # y2_hat corresponds to retargeted_actor = actors[i, 0], original_actor = actors[i, 1]
        anonymized.append({
            'skeleton': y2_hat[i],
            'retargeted_actor': actors[i, 0],
            'original_actor': actors[i, 1],
            'action': actions[i, 1]
        })

    return anonymized

# Eval Models

## Skeleton MixFormer

In [ ]:
def import_class(import_str):
    mod_str, _sep, class_str = import_str.rpartition('.')
    __import__(mod_str)
    try:
        return getattr(sys.modules[mod_str], class_str)
    except AttributeError:
        raise ImportError('Class {} cannot be found ({})'.format(class_str, traceback.format_exc()))

def eval_skeleton_mixformer(paired_test, model, prep_data):
    # Load the action recognition model
    ar_model_weights = f'eval/mixformer/pretrained/{dataset}/ar.pth'
    ar_model_class = 'model.ske_mixf.Model' 
    ar_model_args = {
        'num_class': datasets[dataset]['num_classes'],
        'num_point': 25,
        'num_person': 2,
        'graph': datasets[dataset]['graph'],
    }
    # Import and instantiate the action recognition model
    AR_Model = import_class(ar_model_class)
    ar_model = AR_Model(**ar_model_args)
    # Load the weights
    ar_state_dict = torch.load(ar_model_weights)
    ar_state_dict = {k.replace('module.', ''): v for k, v in ar_state_dict.items()}
    ar_model.load_state_dict(ar_state_dict)
    ar_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ar_model = ar_model.to(ar_device)
    ar_model.eval()

    # Load the re-identification model
    ri_model_weights = f'eval/mixformer/pretrained/{dataset}/ri.pth'
    ri_model_class = 'model.ske_mixf.Model'
    ri_model_args = { # swap num_classes to num_actors
        'num_class': datasets[dataset]['num_classes'],  # Number of actors/subjects in NTU RGB+D
        'num_point': 25,
        'num_person': 2,  # Assuming one person after anonymization
        'graph': datasets[dataset]['graph'],
        # Add other model arguments if necessary
    }
    # Import and instantiate the re-identification model
    RI_Model = import_class(ri_model_class)
    ri_model = RI_Model(**ri_model_args)
    # Load the weights
    ri_state_dict = torch.load(ri_model_weights)
    # Remove 'module.' prefix if necessary
    ri_state_dict = {k.replace('module.', ''): v for k, v in ri_state_dict.items()}
    ri_model.load_state_dict(ri_state_dict)
    ri_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ri_model = ri_model.to(ri_device)
    ri_model.eval()

    total_samples = 0
    correct_action = 0
    correct_actor_ret = 0  # Correctly predicted retargeted actor
    correct_actor_orig = 0  # Correctly predicted original actor
    
    for batch in paired_test:
        # Get the anonymized skeletons
        anonymized_data = get_anonymized_paired(batch, model, prep_data)  # This function should return a list of dictionaries

        # Prepare lists for batch data
        ar_batch_data = []
        ar_labels = []
        ri_batch_data = []
        ri_ret_labels = []  # Retargeted actor labels
        ri_orig_labels = []  # Original actor labels

        # For each anonymized skeleton in the batch
        for item in anonymized_data:
            skeleton = item['skeleton']  # Should be tensor of shape (T, D)
            action_label = item['action']  # The action label (integer)
            retargeted_actor = item['retargeted_actor']
            original_actor = item['original_actor']

            # Prepare data for action recognition model
            # Reshape skeleton to (C, T, V, M), where C=3, V=25, M=2
            T = skeleton.shape[0]
            D = skeleton.shape[1]
            V = 25  # Number of joints
            C = 3   # Number of channels
            M = 2   # Number of persons (model expects 2)

            assert D == V * C, f"Dimension mismatch: D={D}, V*C={V*C}"

            # Reshape skeleton to (T, V, C)
            skeleton = skeleton.view(T, V, C)

            # Expand to include second person (zeros)
            skeleton = skeleton.unsqueeze(1)  # Shape: (T, 1, V, C)
            zeros_tensor = torch.zeros_like(skeleton)  # Shape: (T, 1, V, C)
            # Concatenate along M dimension
            skeleton = torch.cat([skeleton, zeros_tensor], dim=1)  # Shape: (T, 2, V, C)

            # Now permute to get (C, T, V, M)
            skeleton = skeleton.permute(3, 0, 2, 1).contiguous()  # Shape: (C, T, V, M)
            ar_batch_data.append(skeleton.detach().cpu().numpy())
            ar_labels.append(action_label)

            # Prepare data for re-identification model
            # Use the same skeleton data
            ri_batch_data.append(skeleton.detach().cpu().numpy())
            ri_ret_labels.append(retargeted_actor)
            ri_orig_labels.append(original_actor)

        # Convert lists to tensors
        ar_batch_data = np.stack(ar_batch_data)
        ar_labels = np.array(ar_labels)
        ar_batch_data = torch.tensor(ar_batch_data, dtype=torch.float32).to(ar_device)
        ar_labels = torch.tensor(ar_labels, dtype=torch.long).to(ar_device)

        # Evaluate action recognition
        with torch.no_grad():
            ar_output = ar_model(ar_batch_data)
            _, ar_predicted = torch.max(ar_output.data, 1)
            correct_action += (ar_predicted == ar_labels).sum().item()
            total_samples += ar_labels.size(0)

        # Prepare re-identification data
        ri_batch_data = np.stack(ri_batch_data)
        ri_batch_data = torch.tensor(ri_batch_data, dtype=torch.float32).to(ri_device)
        # Convert actor labels to tensors
        ri_ret_labels = np.array(ri_ret_labels)
        ri_ret_labels = torch.tensor(ri_ret_labels, dtype=torch.long).to(ri_device)
        ri_orig_labels = np.array(ri_orig_labels)
        ri_orig_labels = torch.tensor(ri_orig_labels, dtype=torch.long).to(ri_device)

        # Evaluate re-identification
        with torch.no_grad():
            ri_output = ri_model(ri_batch_data)
            _, ri_predicted = torch.max(ri_output.data, 1)
            # Compare predicted actor IDs with retargeted_actor and original_actor
            correct_actor_ret += (ri_predicted == ri_ret_labels).sum().item()
            correct_actor_orig += (ri_predicted == ri_orig_labels).sum().item()

        break  # Only evaluate one batch for now

    # Compute action recognition accuracy
    action_accuracy = correct_action / total_samples * 100
    print(f'Action Recognition Accuracy: {action_accuracy:.2f}%')

    # Print re-identification results
    print(f'Re-identification Results:')
    print(f'Predicted Retargeted Actor: {correct_actor_ret} out of {total_samples} samples ({(correct_actor_ret/total_samples)*100:.2f}%)')
    print(f'Predicted Original Actor: {correct_actor_orig} out of {total_samples} samples ({(correct_actor_orig/total_samples)*100:.2f}%)')
    neither = total_samples - correct_actor_ret - correct_actor_orig
    neither_percentage = (neither / total_samples) * 100
    print(f'Predicted Neither Actor: {neither} out of {total_samples} samples ({neither_percentage:.2f}%)')


# Now, evaluate using the eval_skeleton_mixformer function
eval_skeleton_mixformer(paired_test, model, prep_data)

## Semantic Guided Neural Network (SGN)

In [ ]:
def import_class(import_str):
    mod_str, _sep, class_str = import_str.rpartition('.')
    __import__(mod_str)
    try:
        return getattr(sys.modules[mod_str], class_str)
    except AttributeError:
        raise ImportError('Class {} cannot be found ({})'.format(class_str, traceback.format_exc()))

def eval_skeleton_sgn(paired_test, model, prep_data):
    # Load the action recognition model
    ar_model_weights = f'eval/sgn/pretrained/{dataset}/ar.pth'
    ar_model_class = 'model.sgn.SGN' 
    ar_model_args = {
        'num_classes': datasets[dataset]['num_classes'],
        'dataset': dataset,
        'seg': 20,
    }
    # Import and instantiate the action recognition model
    AR_Model = import_class(ar_model_class)
    ar_model = AR_Model(**ar_model_args)
    # Load the weights
    ar_state_dict = torch.load(ar_model_weights, weights_only=True)['state_dict']
    ar_model.load_state_dict(ar_state_dict)
    ar_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ar_model = ar_model.to(ar_device)
    ar_model.eval()

    # Load the re-identification model
    ri_model_weights = f'eval/sgn/pretrained/{dataset}/ri.pth'
    ri_model_class = 'model.sgn.SGN'
    ri_model_args = {
        'num_classes': datasets[dataset]['num_actors'],
        'dataset': dataset,
        'seg': 20,
    }
    # Import and instantiate the re-identification model
    RI_Model = import_class(ri_model_class)
    ri_model = RI_Model(**ri_model_args)
    # Load the weights
    ri_state_dict = torch.load(ri_model_weights, weights_only=True)['state_dict']
    ri_model.load_state_dict(ri_state_dict)
    ri_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ri_model = ri_model.to(ri_device)
    ri_model.eval()

    total_samples = 0
    correct_action = 0
    correct_actor_ret = 0  # Correctly predicted retargeted actor
    correct_actor_orig = 0  # Correctly predicted original actor
    
    for batch in paired_test:
        # Get the anonymized skeletons
        anonymized_data = get_anonymized_paired(batch, model, prep_data)  # Returns a list of dictionaries

        # Prepare lists for batch data
        ar_batch_data = []
        ar_labels = []
        ri_batch_data = []
        ri_ret_labels = []  # Retargeted actor labels
        ri_orig_labels = []  # Original actor labels

        # For each anonymized skeleton in the batch
        for item in anonymized_data:
            skeleton = item['skeleton']  # Should be tensor of shape (T, D)
            action_label = item['action']  # The action label (integer)
            retargeted_actor = item['retargeted_actor']
            original_actor = item['original_actor']

            # Prepare data for SGN model
            # SGN expects input of shape (batch_size, seg, 150)
            # where 150 = num_person * num_joint * in_channel (e.g., 2 * 25 * 3 = 150)
            # First, we need to segment the sequence into 'seg' segments
            T = skeleton.shape[0]
            D = skeleton.shape[1]  # Should be num_joint * in_channel
            num_joint = V
            in_channel = 3  # Number of channels

            # Ensure D matches num_joint * in_channel
            assert D == num_joint * in_channel, f"Dimension mismatch: D={D}, num_joint*in_channel={num_joint*in_channel}"

            # Reshape skeleton to (T, num_joint, in_channel)
            # Since we have only one person, we need to add a second person with zeros
            skeleton = skeleton.view(T, num_joint, in_channel)
            print(skeleton.shape)

            # Concatenate the data for both persons: (T, 75)
            skeleton = skeleton.view(T, -1)  # Shape: (T, 75)
            print(skeleton.shape, '\n')

            # Segment the sequence into 'seg' segments
            seg = ar_model_args['seg']
            if T < seg:
                # If sequence is shorter than 'seg', pad it
                pad_size = seg - T
                padding = torch.zeros(pad_size, skeleton.shape[1])
                skeleton = torch.cat([skeleton, padding], dim=0)
            else:
                # If sequence is longer, truncate or sample uniformly
                indices = np.linspace(0, T - 1, seg).astype(int)
                skeleton = skeleton[indices, :]

            skeleton = skeleton.unsqueeze(0)  # Shape: (1, seg, 150)
            ar_batch_data.append(skeleton)
            ar_labels.append(int(action_label))

            # Prepare data for re-identification model
            # Use the same skeleton data
            ri_batch_data.append(skeleton)
            ri_ret_labels.append(int(retargeted_actor))
            ri_orig_labels.append(int(original_actor))

        # Stack batch data
        ar_batch_data = torch.cat(ar_batch_data, dim=0)  # Shape: (batch_size, seg, 150)
        ar_labels = torch.tensor(ar_labels, dtype=torch.long)
        ri_batch_data = torch.cat(ri_batch_data, dim=0)  # Shape: (batch_size, seg, 150)
        ri_ret_labels = torch.tensor(ri_ret_labels, dtype=torch.long)
        ri_orig_labels = torch.tensor(ri_orig_labels, dtype=torch.long)

        batch_size = ar_batch_data.size(0)
        total_samples += batch_size

        # Evaluate action recognition
        with torch.no_grad():
            ar_output = ar_model(ar_batch_data.to(ar_device))
            ar_output = ar_output.view((-1, ar_output.size(1)))
            _, ar_predicted = torch.max(ar_output.data, 1)
            correct_action += (ar_predicted.cpu() == ar_labels).sum().item()

        # Evaluate re-identification
        with torch.no_grad():
            ri_output = ri_model(ri_batch_data.to(ri_device))
            ri_output = ri_output.view((-1, ri_output.size(1)))
            _, ri_predicted = torch.max(ri_output.data, 1)
            # Compare predicted actor IDs with retargeted_actor and original_actor
            correct_actor_ret += (ri_predicted.cpu() == ri_ret_labels).sum().item()
            correct_actor_orig += (ri_predicted.cpu() == ri_orig_labels).sum().item()

        break  # Only evaluate one batch for now

    # Compute action recognition accuracy
    action_accuracy = correct_action / total_samples * 100
    print(f'Action Recognition Accuracy: {action_accuracy:.2f}%')

    # Print re-identification results
    print(f'Re-identification Results:')
    print(f'Predicted Retargeted Actor: {correct_actor_ret} out of {total_samples} samples ({(correct_actor_ret/total_samples)*100:.2f}%)')
    print(f'Predicted Original Actor: {correct_actor_orig} out of {total_samples} samples ({(correct_actor_orig/total_samples)*100:.2f}%)')
    neither = total_samples - correct_actor_ret - correct_actor_orig
    neither_percentage = (neither / total_samples) * 100
    print(f'Predicted Neither Actor: {neither} out of {total_samples} samples ({neither_percentage:.2f}%)')


# Now, evaluate using the eval_skeleton_mixformer function
eval_skeleton_sgn(paired_test, model, prep_data)